# 03_transformations — Typage, aplatissement, colonnes dérivées
Relecture des tables `brz_`, application des transformations en DataFrame API uniquement (ni Pandas, ni SQL), puis réécriture de chaque table à la place de l'originale.

Principe de contrôle appliqué à chaque table :
1. compter les NULL avant conversion ;
2. convertir ;
3. compter les NULL après : tout écart = conversion ratée (ex. `to_date` avec un mauvais format renvoie NULL sans erreur) ;
4. vérifier que le nombre de lignes n'a pas changé.

Pourquoi `overwriteSchema` ? Réécrire une table avec des types différents modifie son schéma. Delta refuse par défaut (*schema enforcement*) : cela protège la table contre une écriture accidentelle qui casserait les requêtes en aval. Ici le changement est voulu, on l'autorise donc explicitement, table par table.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import types as T

TARGET = "alpenia_dev.bronze"
# Les horodatages sont en UTC : on fixe la session en UTC pour lire et afficher sans décalage
spark.conf.set("spark.sql.session.timeZone", "UTC")

stats = []   # chiffres consignés pour le rapport

def null_counts(df, cols):
    return df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in cols]).first().asDict()

def save(df_before, df_after, table, checked_cols):
    """Contrôle lignes + NULL, puis réécrit la table Delta avec son nouveau schéma."""
    n_before, n_after = df_before.count(), df_after.count()
    nb = null_counts(df_before, [c for c in checked_cols if c in df_before.columns])
    na = null_counts(df_after, checked_cols)
    for c in checked_cols:
        stats.append((table, c, nb.get(c), na[c]))
    assert n_before == n_after, f"{table} : {n_before} -> {n_after} lignes"
    unexpected = {c: (nb[c], na[c]) for c in nb if na[c] != nb[c]}
    assert not unexpected, f"{table} : NULL apparus à la conversion {unexpected}"
    (df_after.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(f"{TARGET}.{table}"))
    print(f"{table} : {n_after:,} lignes, réécrite")
    spark.table(f"{TARGET}.{table}").printSchema()

## 1. contracts
`start_date` est fournie en `dd/MM/yyyy` : il faut donner le format à `to_date`, sinon le résultat est NULL.

In [ ]:
src = spark.table(f"{TARGET}.brz_contracts")
df = src.select(
    F.col("contract_id").cast("string"),
    F.col("client_id").cast("string"),
    F.col("product").cast("string"),
    F.col("city").cast("string"),
    F.to_date("start_date", "dd/MM/yyyy").alias("start_date"),
    F.col("annual_premium_eur").cast("double"),
    F.col("status").cast("string"),
)
save(src, df, "brz_contracts", ["start_date", "annual_premium_eur"])

## 2. clients
`signup_date` est déjà au format ISO `yyyy-MM-dd`.

In [ ]:
src = spark.table(f"{TARGET}.brz_clients")
df = src.select(
    F.col("client_id").cast("string"),
    F.to_date("signup_date", "yyyy-MM-dd").alias("signup_date"),
    F.col("city").cast("string"),
    F.col("segment").cast("string"),
    F.col("birth_year").cast("int"),
)
save(src, df, "brz_clients", ["signup_date", "birth_year"])

## 3. pricing
Montants en `DECIMAL` (valeur exacte, pas d'arrondi binaire comme avec `DOUBLE`) : `DECIMAL(7,2)` jusqu'à 99 999,99 €, `DECIMAL(5,2)` jusqu'à 999,99.

In [ ]:
src = spark.table(f"{TARGET}.brz_pricing")
df = src.select(
    F.col("city").cast("string"),
    F.col("product").cast("string"),
    F.col("base_premium_eur").cast(T.DecimalType(7, 2)).alias("base_premium_eur"),
    F.col("rate_per_1k_eur").cast(T.DecimalType(5, 2)).alias("rate_per_1k_eur"),
    F.to_date("valid_from", "yyyy-MM-dd").alias("valid_from"),
)
save(src, df, "brz_pricing", ["base_premium_eur", "rate_per_1k_eur", "valid_from"])

## 4. claims
Le Parquet est déjà typé (horodatages `TIMESTAMP` UTC, montants `DOUBLE`) : on force tout de même les types cibles pour garantir le contrat de schéma.

Colonne dérivée `settlement_days` : écart en secondes entre déclaration et règlement ÷ 86 400, arrondi à 1 décimale.

In [ ]:
src = spark.table(f"{TARGET}.brz_claims")
df = (src.select(
        F.col("claim_id").cast("string"),
        F.col("contract_id").cast("string"),
        F.col("client_id").cast("string"),
        F.col("declared_ts").cast("timestamp"),
        F.col("settled_ts").cast("timestamp"),
        F.col("peril").cast("string"),
        F.col("amount_eur").cast("double"),
        F.col("deductible_eur").cast("double"),
    )
    .withColumn("settlement_days",
        F.round((F.unix_timestamp("settled_ts") - F.unix_timestamp("declared_ts")) / 86400, 1)))
save(src, df, "brz_claims", ["declared_ts", "settled_ts", "amount_eur", "settlement_days"])

## 5. web_events
- `event_ts` : texte ISO 8601 avec suffixe `Z` (= UTC). Le motif `X` du format interprète ce `Z` comme un fuseau.
- **Aplatissement** : la colonne `STRUCT` `quote` est remplacée par `quote_product` et `quote_premium_eur`. On ne garde dans le `select` que les colonnes voulues : la structure disparaît de la table finale.

In [ ]:
src = spark.table(f"{TARGET}.brz_web_events")

if "quote" in src.columns:
    quote_product = F.col("quote.product")
    quote_premium = F.col("quote.premium_eur")
else:   # notebook relancé sans repasser par 02_ingestion : déjà aplati
    quote_product = F.col("quote_product")
    quote_premium = F.col("quote_premium_eur")

df = src.select(
    F.col("event_id").cast("string"),
    F.col("client_id").cast("string"),
    F.to_timestamp(F.col("event_ts").cast("string"), "yyyy-MM-dd'T'HH:mm:ssX").alias("event_ts")
        if dict(src.dtypes)["event_ts"] == "string" else F.col("event_ts"),
    quote_product.cast("string").alias("quote_product"),
    quote_premium.cast("double").alias("quote_premium_eur"),
    F.col("channel").cast("string"),
    F.col("action").cast("string"),
)
save(src, df, "brz_web_events", ["event_ts", "quote_product", "quote_premium_eur"])
assert "quote" not in spark.table(f"{TARGET}.brz_web_events").columns

## 6. expertises
- `opened_at` / `closed_at` : texte `yyyy-MM-dd HH:mm:ss` ; `closed_at` est NULL pour les expertises en cours. Ces NULL sont attendus : le contrôle vérifie qu'il n'y en a pas plus après conversion qu'avant.
- `items_replaced` reste un `ARRAY<STRING>` (éventuellement vide).
- Colonnes dérivées : `items_count` (taille du tableau, 0 si vide) et `is_open` (`true` si `closed_at` est NULL).

In [ ]:
src = spark.table(f"{TARGET}.brz_expertises")
df = (src.select(
        F.col("expertise_id").cast("string"),
        F.col("claim_id").cast("string"),
        F.to_timestamp(F.col("opened_at").cast("string"), "yyyy-MM-dd HH:mm:ss").alias("opened_at"),
        F.to_timestamp(F.col("closed_at").cast("string"), "yyyy-MM-dd HH:mm:ss").alias("closed_at"),
        F.col("finding_type").cast("string"),
        F.col("items_replaced").cast("array<string>"),
        F.col("cost_eur").cast("double"),
    )
    .withColumn("items_count",
        F.when(F.col("items_replaced").isNull(), F.lit(0)).otherwise(F.size("items_replaced")))
    .withColumn("is_open", F.col("closed_at").isNull()))
save(src, df, "brz_expertises", ["opened_at", "closed_at", "cost_eur", "items_count"])

## 7. Contrôle global
NULL avant / après conversion pour chaque colonne contrôlée, et nombre de lignes final de chaque table. Ces chiffres alimentent le rapport.

In [ ]:
display(spark.createDataFrame(stats, "table STRING, column STRING, nulls_before LONG, nulls_after LONG"))

In [ ]:
EXPECTED_ROWS = {"brz_contracts": 4000, "brz_clients": 3000, "brz_pricing": 6,
                 "brz_claims": 24000, "brz_web_events": 90000, "brz_expertises": 450}
final = [(t, e, spark.table(f"{TARGET}.{t}").count()) for t, e in EXPECTED_ROWS.items()]
display(spark.createDataFrame(final, "table STRING, expected_rows LONG, actual_rows LONG"))
assert all(e == a for _, e, a in final)